In [1]:
# installo il repo
%pip install ucimlrepo

Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys

# Importo moduli Scikit-Learn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

In [3]:
# CELLA DELLE FUNZIONI
# Imposto lo stile dei grafici
plt.style.use('seaborn-v0_8-whitegrid')


# -------------------------------------------------------------------
# FUNZIONI DI SUPPORTO (Caricamento e Analisi Componenti)
# Questa funzione nasce con l'importazione del dataset locale scarcato dal repo online, 
# successivamente è stata modificata per caricare il dataset
# direttamente dal repo di UCI. 
# ho preferito rendere gestire l'eccezione del file mancante per velocizzare test locali
# e rendere lo script indipendente da file locali.
# Durante tale cambiamento apparentemente poco impattante mi
# sono reso conto che i nomi delle colonne avevano subito una variazione
# per cui ho provveduto a normalizzarli nel file 
def load_and_clean_data(file_path):
    """Gestisce il caricamento robusto (Locale -> Online) e la pulizia iniziale."""
    try:
        # TENTATIVO DI CARICAMENTO FILE LOCALE
        print(f"Tentativo di caricamento da: {file_path}...")
        df = pd.read_csv(file_path)
        print("Dataset locale caricato con successo!")

        X = df.drop(['name', 'status'], axis=1)
        y = df['status']

    except FileNotFoundError:
        # TENTATIVO DI CARICAMENTO DAL REPO UCIML (Fallback)
        print("File locale non trovato. Download da UCI ML Repository...")
        try:
            from ucimlrepo import fetch_ucirepo
            parkinsons = fetch_ucirepo(id=174)
            X = parkinsons.data.features
            y_raw = parkinsons.data.targets


            # Gestione dimensionale del target squeeze appiattisce i dati
            y = y_raw.squeeze()
            y.name = 'status'
            # Ricostruisco df per compatibilità
            df = pd.concat([X, y], axis=1)

            # Rinomina colonne per standardizzazione
            nomi_originali = [
                'MDVP:Fo(Hz)', 'MDVP:Fhi(Hz)', 'MDVP:Flo(Hz)', 'MDVP:Jitter(%)',
                'MDVP:Jitter(Abs)', 'MDVP:RAP', 'MDVP:PPQ', 'Jitter:DDP',
                'MDVP:Shimmer', 'MDVP:Shimmer(dB)', 'Shimmer:APQ3', 'Shimmer:APQ5',
                'MDVP:APQ', 'Shimmer:DDA', 'NHR', 'HNR', 'RPDE', 'DFA',
                'spread1', 'spread2', 'D2', 'PPE', 'status'
            ]
            # verifico le colonne e le rinomino
            if len(df.columns) == len(nomi_originali):
                df.columns = nomi_originali
                # Elimino da X la feature target
                X = df.drop(['status'], axis=1)

            print("Dataset online scaricato e formattato.")

        except Exception as e:
            print(f"ERRORE CRITICO: {e}")
            sys.exit(1)
    print ("\nStampo a schermo le colonne X:")
    print (X.columns)
    print ("\nStampo a schermo le colonne (Target) y:")
    print (y.name)


    return df, X, y

# -------------------------------------------------------------------
# Analisi dei dati grezzi PC -> Tabella Esempio PCA Dati non trasposti

def raw_components(componenti, feature_names):
    """**************
    Per verificare come opera la PCA creo una funzione che restituisce
    i dati grezzi non trasposti.
    **************"""
    return pd.DataFrame(componenti, columns=feature_names, index=['PC1', 'PC2'])


# -------------------------------------------------------------------
# Analisi della PCA -> Tabella Analisi PC

def analizza_componente_dinamica(nome_pc, series_loadings):
    """**************
    Per capire come opera la PCA analizzo i dati in modo dinamico.
    Stampo a schermo le feature che superano la soglia media di influenza.
    **************"""
    # Calcolo la media dei pesi (in valore assoluto)
    soglia_dinamica = series_loadings.abs().mean()

    # Seleziono le features rilevanti ovvero quelle che superano la media calcolata sopra
    feature_rilevanti = series_loadings[series_loadings.abs() > soglia_dinamica]

    # Stampa i risultati
    # .4f significa restituiscimi dopo la virgola del valore 4 cifre del numero float
    # e trattale come un normale numero decimale
    print("\n" + "="*50)
    print(f"|| ANALISI {nome_pc} ||")
    print("="*50)    
    print(f"Soglia calcolata (Media pesi): {soglia_dinamica:.4f}")
    print(f"Numero feature rilevanti: {len(feature_rilevanti)} su {len(series_loadings)}")
    print("\nTop Feature Rilevanti (ordinate per importanza):")
    print(feature_rilevanti.abs().sort_values(ascending=False))

# -------------------------------------------------------------------
# FUNZIONE PCA

def run_pca_analysis(X_scaled, y, feature_names):
    """Esegue PCA, stampa l'analisi dei loadings e genera lo scatter plot."""
    print("\n" + "="*50)
    print("PCA E VISUALIZZAZIONE")
    print("="*50)

    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_scaled)

    # Richiamo e stampo i dati grezzi
    dat_grezzi = raw_components(pca.components_, feature_names)
    print("\n" + "="*50)
    print(f"|| DATI GREZZI PRIMA DI TRASPORRE PC1 ||")
    print("="*50)    
    print(dat_grezzi.loc[['PC1'], :].iloc[:, :])

    print("\n" + "="*50)
    print(f"|| DATI GREZZI PRIMA DI TRASPORRE PC2 ||")
    print("="*50)    
    print(dat_grezzi.loc[['PC2'], :].iloc[:, :])

    # Creo la tabella dei pesi (Loadings) trasposta
    loadings = pd.DataFrame(pca.components_.T, columns=['PC1', 'PC2'], index=feature_names)

    print("\n" + "="*50)
    print(f"|| FEATURE CHE CREANO LA PC1 DATI TRASPOSTI (ASSE X)||")
    print("="*50)    
    print(loadings['PC1'].abs().sort_values(ascending=False))

    print("\n" + "="*50)
    print(f"|| FEATURE CHE CREANO LA PC2 DATI TRASPOSTI (ASSE Y)||")
    print("="*50)    
    print(loadings['PC2'].abs().sort_values(ascending=False))

    # Analisi dei componenti trasposti
    analizza_componente_dinamica("PC1", loadings['PC1'])
    analizza_componente_dinamica("PC2", loadings['PC2'])

    # Visualizzazione Scatter Plot della PCA
    plt.figure(figsize=(10, 7))
    # Colore in base allo status (gestione sicura del formato y)
    c_values = y.values.ravel() if isinstance(y, (pd.DataFrame, pd.Series)) else y

    scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=c_values, cmap='coolwarm', edgecolor='k', alpha=0.7)
    plt.xlabel('Principal Component 1 (Patologia)')
    plt.ylabel('Principal Component 2 (Fisiologia)')
    plt.title('Proiezione PCA del Dataset Parkinson')
    plt.colorbar(scatter, label='Status (0=Sano, 1=Parkinson)')

    # Se voglio l'immagine decommento plt.savefig
    #plt.savefig('pca_parkinsons.png', dpi=300)
    #print("\n Grafico PCA salvato come 'pca_parkinsons.png'")
    plt.show() # Decommentare se si usa Jupyter interattivo





# -------------------------------------------------------------------

def seleziona_modelli(scelta):
    """FUNZIONE PRINCIPALE MODELLI
    Questa funzione permette in base ad una scelta in input 0 oppure 1
    Restituisce il dizionario dei modelli configurato in base all'input utente.
    0: Baseline (Parametri standard)
    1: TEST IPERPARAMETRI (Parametri per sbilanciamento e small-data)
    """
    if scelta == 1:
        print("\n" + "*"*60)
        print(" MODALITÀ ATTIVA: (TEST IPERPARAMETRI)")
        print("*"*60)
        """blocco per editing degli iperparamentri nella modalità test 
        iperparametri avviabile inserendo il valore 1 in input"""
        
        return {
            "Logistic Regression": LogisticRegression(
                solver='liblinear', class_weight='balanced', random_state=42
            ),
            "SVM": SVC(
                #kernel='rbf', probability=True, C=1.0, gamma='scale', random_state=42
                kernel='rbf', probability=True, C=0.5, gamma='auto', random_state=42
            ),
            "Random Forest": RandomForestClassifier(
                n_estimators=200, max_depth=10, class_weight='balanced', random_state=42
                #n_estimators=300, max_depth=6, min_samples_leaf=4, class_weight='balanced', random_state=42
            ),
            "Deep Learning (MLP)": MLPClassifier(
                hidden_layer_sizes=(64, 32), activation='tanh', solver='lbfgs',
                max_iter=1000, alpha=0.0001, random_state=42
            )
        }
    else:
        print("\n" + "-"*60)
        print(" MODALITÀ ATTIVA: BASELINE")
        print(" - Parametri di default Scikit-Learn (migliore risultato nonchè quello utilizzato nell'elaborato)")
        print("-" * 60)
        return {
            "Logistic Regression": LogisticRegression(random_state=42),
            "SVM": SVC(probability=True, random_state=42),
            "Random Forest": RandomForestClassifier(random_state=42),
            "Deep Learning (MLP)": MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42)
        }




# -------------------------------------------------------------------
# Funzione di addestramento dei modelli
#
def run_models_evaluation(X_train, X_test, y_train, y_test, scelta_modelli):
    """Addestro i modelli, stampo metriche e genero grafici comparativi (ROC e Matrici)."""
    print("\n" + "="*50)
    print("TRAINING E VALUTAZIONE MODELLI")
    print("="*50)

    models = seleziona_modelli(scelta_modelli)

    confusion_matrices = []
    model_names = []

    # Inizializzo figura per curve ROC
    plt.figure(figsize=(10, 8))

    for name, model in models.items():
        print(f"\n🔹 Addestramento {name}...")
        model.fit(X_train, y_train)

        # Predizioni
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1] # Probabilità classe positiva

        # Metriche
        print(f"--- Report {name} ---")
        print(classification_report(y_test, y_pred))

        roc_auc = roc_auc_score(y_test, y_prob)
        print(f"ROC AUC Score: {roc_auc:.4f}")

        # Salvataggio dati per grafici
        cm = confusion_matrix(y_test, y_pred)
        confusion_matrices.append(cm)
        model_names.append(name)

        # Plot Curva ROC (sullo stesso grafico)
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.2f})', linewidth=2)

    # Finalizzo Grafico ROC
    plt.plot([0, 1], [0, 1], 'k--', label='Random Guess')
    plt.xlabel('False Positive Rate (1 - Specificity)')
    plt.ylabel('True Positive Rate (Sensitivity)')
    plt.title('Confronto Curve ROC: Capacità Diagnostica')
    plt.legend(loc='lower right')
    plt.grid(True, alpha=0.3)
    #plt.savefig('roc_comparison.png', dpi=300)
    #print("Grafico ROC salvato come 'roc_comparison.png'")
    print("\n" + "="*50)
    print(f"|| GRAFICO ROC||")
    print("="*50)    
    plt.show()

    # Plot Matrici di Confusione (2x2)
    fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(14, 12))
    axes = axes.flatten()

    for i, cm in enumerate(confusion_matrices):
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i], cbar=False, annot_kws={"size": 14})
        axes[i].set_title(f'Confusion Matrix - {model_names[i]}', fontsize=14, fontweight='bold')
        axes[i].set_xlabel('Predetto', fontsize=12)
        axes[i].set_ylabel('Reale', fontsize=12)
        axes[i].set_xticklabels(['Sano', 'Parkinson'])
        axes[i].set_yticklabels(['Sano', 'Parkinson'])

    plt.tight_layout()
    #plt.savefig('all_confusion_matrices.png', dpi=300)
    #print("Grafico Matrici Confusione salvato come 'all_confusion_matrices.png'")
    print("\n" + "="*50)
    print(f"|| GRAFICO MATRICI CONFUSIONE COMPARATE||")
    print("="*50)
    plt.show() 





# -------------------------------------------------------------------
# FUNZIONE ANALISI CORRELAZIONE
#
def plot_correlation_heatmap(df):
    """Genera la heatmap di correlazione per le feature selezionate."""
    print("\n" + "="*50)
    print("MATRICE DI CORRELAZIONI (FEATURE PIU' IMPATTANTI)")
    print("="*50)

    # Calcolo la correlazione solo su alcune feature chiave per l'esempio
    cols_to_plot = [
        'MDVP:Fo(Hz)', 'MDVP:Jitter(%)', 'MDVP:RAP', 'MDVP:Shimmer',
        'NHR', 'HNR', 'status', 'PPE', 'DFA', 'spread1', 'spread2', 'D2'
    ]

    # Verifica esistenza colonne (intersezione sicura)
    valid_cols = [c for c in cols_to_plot if c in df.columns]

    subset = df[valid_cols]
    corr_matrix = subset.corr()

    plt.figure(figsize=(10, 8))
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
    plt.title("Matrice di Correlazione (Feature Selezionate)")
    #plt.savefig('correlation_matrix.png', dpi=300)
    #print("Heatmap Correlazione salvata come 'correlation_matrix.png'")
    plt.show()

In [6]:
# -------------------------------------------------------------------
# MAIN EXECUTION
#
if __name__ == "__main__":
    # 1. Caricamento Dati
    file_path = "dataset/parkinsons.data"
    df, X, y = load_and_clean_data(file_path)

    # Scaling
    # standardizzo i dati vista l'eterogeneità utilizzando il modulo StandardScaler()
    # che riconduce tutte le feature ad avere media 0 e varianza globale 1
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Split
    # divido il dataset in Training (70%) e Test (30%) test_size=0.3
    # stratify=y serve ad avere una proporzione omogenea tra le classi parkinson e sano
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.3, random_state=42, stratify=y
    )

    # Richiamo i moduli
    # Messaggio di selezione del modello:
    # 0/nessuna risposta=carica il default (soluzione più performante)
    # soluzione migliore
    # 1 = carica un preset di iperparametri teoricamente migliorati ma praticamente non performanti
    print("\n" + "="*40)
    print("SELEZIONE CONFIGURAZIONE MODELLI")
    print("="*40)
    print(" [0] BASELINE    -> Parametri standard")
    print(" [1] OTTIMIZZATA -> Parametri Tuned")

    try:
        user_input = input("\nInserisci la tua scelta (0 o 1): ")
        scelta = int(user_input)

        # Controllo extra se inserisce un numero diverso da 0 o 1
        if scelta not in [0, 1]:
            print("Numero non valido. Verrà usata la configurazione BASELINE (0).")
            scelta = 0

    except ValueError:
        print("Input non valido (non è un numero). Verrà usata la configurazione BASELINE (0).")
        scelta = 0

    # Passo la variabile 'scelta' alla funzione principale


    run_pca_analysis(X_scaled, y, X.columns)
    plot_correlation_heatmap(df)

    run_models_evaluation(X_train, X_test, y_train, y_test, scelta_modelli=scelta)


    print("\nElaborazione Completata con Successo.")

Tentativo di caricamento da: dataset/parkinsons.data...
Dataset locale caricato con successo!

Stampo a schermo le colonne X:
Index(['MDVP:Fo(Hz)', 'MDVP:Fhi(Hz)', 'MDVP:Flo(Hz)', 'MDVP:Jitter(%)',
       'MDVP:Jitter(Abs)', 'MDVP:RAP', 'MDVP:PPQ', 'Jitter:DDP',
       'MDVP:Shimmer', 'MDVP:Shimmer(dB)', 'Shimmer:APQ3', 'Shimmer:APQ5',
       'MDVP:APQ', 'Shimmer:DDA', 'NHR', 'HNR', 'RPDE', 'DFA', 'spread1',
       'spread2', 'D2', 'PPE'],
      dtype='object')

Stampo a schermo le colonne (Target) y:
status

SELEZIONE CONFIGURAZIONE MODELLI
 [0] BASELINE    -> Parametri standard
 [1] OTTIMIZZATA -> Parametri Tuned


KeyboardInterrupt: Interrupted by user

In [ ]:
0
